In [ ]:
import sys
from pyprojroot import here

# Ritorna il percorso assoluto della root del progetto
PROJECT_ROOT = str(here()) + "/"
sys.path.append(PROJECT_ROOT)
sys.dont_write_bytecode = True

print(f"La root del progetto è: {PROJECT_ROOT}")

In [ ]:
from paths import INV_PATH, TP2_PATH, MODEL_PATH, COLUMN_PATH

ABS_PATH = PROJECT_ROOT + INV_PATH + TP2_PATH
LOAD_MODEL = PROJECT_ROOT + MODEL_PATH + COLUMN_PATH

model_name = "column.blend"
load_data_bound = ABS_PATH + "./files/data.csv"

load_initial_points = ABS_PATH + "files/initial.csv"
load_boundary_points = ABS_PATH + "files/gamma.csv"
load_collocation_points = ABS_PATH + "files/omega.csv"

fig_dir = "figures/"
loss_fig = ABS_PATH + fig_dir + "loss.png"
param_fig = ABS_PATH + fig_dir + "param.png"

params_save = ABS_PATH + "files/pred_parameters.csv"

In [ ]:
# Import
import torch
import pandas as pd
import matplotlib.pyplot as plt

from modelaquisition.bl2pina import Blend2Pina
from pina.condition import Condition
from pina.solvers.pinns import RBAPINN
from pina.equation import SystemEquation
from pina.callbacks import MetricTracker
from pina.geometry import CartesianDomain
from pina.model import ResidualFeedForward
from pina.operators import laplacian, grad
from pina import LabelTensor, Trainer, Plotter
from pytorch_lightning.callbacks import Callback, StochasticWeightAveraging
from pina.problem import SpatialProblem, InverseProblem, TimeDependentProblem

In [ ]:
torch.set_default_dtype(torch.float64)

In [ ]:
# Network variables
lear_rate = 5e-4
swa_lr = 5e-5
decay_rt = 1e-8

# Solver variables
epochs=10_000
batch=None
acc_str = 'gpu'

ipt_var = 4
out_var = 2
lay = 2
neur = 400

# Num points
int_points = 1_000
bound_points = 400
init_points = 400

In [ ]:
column = Blend2Pina(LOAD_MODEL + model_name)

column_int = column.intern(time_interval=[0, 1])
column_bound = column.boundary(time_interval=[0, 1])
column_initial = column.intern(time_interval=[0, 1])

In [ ]:
df = pd.read_csv(load_data_bound, sep=";", index_col=0)

input_pts = df.iloc[:, :4].values
output_pts = df.iloc[:, 4:].values

input_pts = LabelTensor(
    x=torch.tensor(input_pts, dtype=torch.float64),
    labels=['x', 'y', 'z', 't']
)
output_pts = LabelTensor(
    x=torch.reshape(torch.tensor(output_pts, dtype=torch.float64), (output_pts.shape[0], 2)),
    labels=['u1', 'u2']
)

In [ ]:
class ColumnParabolic(SpatialProblem, TimeDependentProblem, InverseProblem):

    input_variables = ['x','y','z','t']
    output_variables = ['u1','u2']
    spatial_domain = column_int.spatial_domain
    temporal_domain = column_int.temporal_domain
    # Definiamo il range per i parametri
    unknown_parameter_domain = CartesianDomain(
        {
            'lambda' : [0, 1],
            'alpha' : [0, 1],
            'beta' : [0, 1]
        }
    )

    @staticmethod
    def residual_u1(input_, output_, params_):
        u_t = grad(output_, input_, components=['u1'], d=['t'])
        lap_u = laplacian(output_=output_, input_=input_, components=['u1'], d=['x', 'y', 'z'])
        force_term = params_['lambda']*torch.exp(params_['lambda']*input_.extract('t'))
        return u_t - lap_u - force_term
    
    @staticmethod
    def residual_u2(input_, output_, params_):
        u_t = grad(output_, input_, components=['u2'], d=['t'])
        lap_u = laplacian(output_=output_, input_=input_, components=['u2'], d=['x', 'y', 'z'])
        force_term = params_['lambda']*torch.exp(params_['lambda']*input_.extract('t')) - 2 * (params_['alpha'] + params_['beta'] + 1)
        return u_t - lap_u - force_term
    
    @staticmethod
    def boundary_u1(input_, output_, params_):
        ref = (
            torch.exp(params_['lambda'] * input_.extract('t')) + (
                params_['alpha']*input_.extract('x') + params_['beta']*input_.extract('y') + input_.extract('z')
            )
        )
        return output_.extract('u1') - ref
    
    @staticmethod
    def boundary_u2(input_, output_, params_):
        ref = (
            torch.exp(params_['lambda'] * input_.extract('t')) + (
                params_['alpha']*(input_.extract('x')**2) + params_['beta']*(input_.extract('y')**2) + input_.extract('z')**2
            )
        )
        return output_.extract('u2') - ref
    
    @staticmethod
    def initial_u1(input_, output_, params_):
        ref = (
            1. + (
                params_['alpha']*input_.extract('x') + params_['beta']*input_.extract('y') + input_.extract('z')
            )
        )
        return output_.extract('u1') - ref
    
    @staticmethod
    def initial_u2(input_, output_, params_):
        ref = (
            1. + (
                params_['alpha']*(input_.extract('x')**2) + params_['beta']*(input_.extract('y')**2) + input_.extract('z')**2
            )
        )
        return output_.extract('u2') - ref
    
    conditions = {
        'Omega' : Condition(
            location=column_int,
            equation=SystemEquation([residual_u1,residual_u2])
        ),
        'Gamma' : Condition(
            location=column_bound,
            equation=SystemEquation([boundary_u1,boundary_u2])
        ),
        'initial' : Condition(
            location=column_initial,
            equation=SystemEquation([initial_u1,initial_u2])
        ),
        'data' : Condition(
            input_points=input_pts.extract(['x','y','z','t']),
            output_points=output_pts.extract(['u1','u2'])
        )
    }
            

In [ ]:
problem = ColumnParabolic()

try:    
    df_omega = pd.read_csv(load_collocation_points, sep=";", index_col=0)
    df_gamma = pd.read_csv(load_boundary_points, sep=";", index_col=0)
    df_initial = pd.read_csv(load_initial_points, sep=";", index_col=0)

    problem.discretise_domain(
        1,
        locations=["Omega", "Gamma", "initial"]
    )
    problem.input_pts["Omega"] = LabelTensor(
        torch.tensor(df_omega.values),
        labels=["t", "x", "y", "z"]
    )
    problem.input_pts["Gamma"] = LabelTensor(
        torch.tensor(df_gamma.values),
        labels=["t", "x", "y", "z"]
    )
    problem.input_pts["initial"] = LabelTensor(
        torch.tensor(df_initial.values),
        labels=["t", "x", "y", "z"]
    )
except:
    problem.discretise_domain(
        n=1000,
        mode='random',
        locations=['Omega']
    )

    problem.discretise_domain(
        n=400,
        mode='random',
        locations=['Gamma', 'initial']
    )

    df_omega = pd.DataFrame(
        problem.input_pts['Omega'].tensor.detach().numpy(),
        columns=['t','x','y','z']
    )

    df_gamma = pd.DataFrame(
        problem.input_pts['Gamma'].tensor.detach().numpy(),
        columns=['t','x','y','z']
    )

    df_initial = pd.DataFrame(
        problem.input_pts['initial'].tensor.detach().numpy(),
        columns=['t','x','y','z']
    )

    df_omega.to_csv(load_collocation_points, sep=";")
    df_gamma.to_csv(load_boundary_points, sep=";")
    df_initial.to_csv(load_initial_points, sep=";")

In [ ]:
class HardMLP(torch.nn.Module):

    def __init__(self, *args, **kwargs):
        super().__init__()
        self.layers = ResidualFeedForward(*args, **kwargs)

    # Nel metodo forward implementiamo il vincolo rigido
    def forward(self, x):
        return  self.layers(x)

In [ ]:
# directory temporanea per salvare i log del training
tmp_dir = ABS_PATH + "column_parabolic_inverse"

class SaveParameters(Callback):
    """
    Callback per salvare i parametri del modello ogni 100 epoche.
    """
    def on_train_epoch_end(self, trainer, _):
        if trainer.current_epoch % 100 == 99:
            torch.save(
                trainer.solver.problem.unknown_parameters,
                '{}/parameters_epoch{}'.format(tmp_dir, trainer.current_epoch)
            )

In [ ]:
# Modello
model = HardMLP(
    input_dimensions=ipt_var,
    output_dimensions=out_var,
    n_layers=lay,
    inner_size=neur
)
# Solver
pinn=RBAPINN(
    problem=problem,
    model=model,
    optimizer_kwargs={
        'lr' : lear_rate,
        'weight_decay' : decay_rt
    },
)

# Trainer
trainer=Trainer(
    solver=pinn,
    max_epochs=epochs,
    batch_size=None,
    accelerator=acc_str,
    precision='64-true',
    callbacks=[SaveParameters(), MetricTracker(), StochasticWeightAveraging(swa_lrs=swa_lr)]
)

# Addestramento
trainer.train()

In [ ]:
my_pl = Plotter()

my_pl.plot_loss(
    trainer=trainer,
    metrics=['Omega_loss'],
    label='Omega_loss',
    logy=True
)

my_pl.plot_loss(
    trainer=trainer,
    metrics=['Gamma_loss'],
    label='Gamma_loss',
    logy=True
)

my_pl.plot_loss(
    trainer=trainer,
    metrics=['data_loss'],
    label='data_loss',
    logy=True
)

my_pl.plot_loss(
    trainer=trainer,
    metrics=['initial_loss'],
    label='initial_loss',
    logy=True
)

plt.savefig(loss_fig, transparent=True)
plt.show()

In [ ]:
epochs_saved = range(99, epochs, 100)
parameters = torch.empty(
    size=(int(epochs/100), 3)
)

for i, epoch in enumerate(epochs_saved):
    params_torch = torch.load('{}/parameters_epoch{}'.format(tmp_dir, epoch))
    for e, var in enumerate(pinn.problem.unknown_variables):
        parameters[i, e] = params_torch[var].data

pred_alpha, pred_beta, pred_lambda = parameters[-1, :]

# Grafico dei parametri
plt.close()
plt.plot(epochs_saved, parameters[:, 2], label='lambda', marker='o')
plt.plot(epochs_saved, parameters[:, 0], label='alpha', marker='s')
plt.plot(epochs_saved, parameters[:, 1], label='beta', marker='^')
plt.ylim(0, 1)
plt.grid()
plt.legend()
plt.xlabel("Epochs")
plt.ylabel("Parameters")
plt.savefig(param_fig, transparent=True)
plt.show()

In [ ]:
par_lambda = torch.tensor(.1)
par_alpha = torch.tensor(.2)
par_beta = torch.tensor(.5)

err_rel_lambda = torch.norm(pred_lambda-par_lambda)/torch.norm(par_lambda)
err_rel_alpha = torch.norm(pred_alpha-par_alpha)/torch.norm(par_alpha)
err_rel_beta = torch.norm(pred_beta-par_beta)/torch.norm(par_beta)

print("RELATIVE ERRORS")
print(f"lambda: {err_rel_lambda.item(): .2e}")
print(f"alpha: {err_rel_alpha.item(): .2e}")
print(f"beta: {err_rel_beta.item(): .2e}")

In [ ]:
df = pd.DataFrame(
    data=[[pred_lambda.item()], [pred_alpha.item()], [pred_beta.item()]],
    columns=["predictions"],
    index=["lambda", "alpha", "beta"]
)

df.to_csv(params_save, sep=";")

In [ ]:
torch.save(model, ABS_PATH + f"models/model_L{lay}_N{neur}_EP{epochs}.pth")